<br><h1>Fake News Detection Part 2</h1><br>

In [10]:
# libraries for part 1
import polars as pl
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer
from sklearn.model_selection import train_test_split
import re
import sys

pd.set_option('display.max_columns', 200)
plt.style.use("ggplot")

# Libraries for part 2
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
import scipy.sparse
import joblib
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.metrics import f1_score, accuracy_score, classification_report
from sklearn.preprocessing import StandardScaler

In [2]:
df = pl.read_csv("./data/995,000_rows_preprocessed.csv", n_rows=20_000).to_pandas()

df = df[['type', 'content']].rename(columns={'type':'isfake'})

real_labels = ['reliable', 'political']
drop_labels = ['unknown', 'satire', 'bias', 'clickbait', 'state', 'hate', 'nan', '2018-02-10 13:43:39.521661']

df = df.loc[~df['isfake'].isin(drop_labels)]

df['isfake'] = df['isfake'].map(lambda x : 0 if x in real_labels else 1)


n = df.shape[0]
indices = np.arange(n)
labels = df['isfake'].to_numpy()

# First split: 80 % train, 20 % temp (will become val + test)
train_idx, temp_idx = train_test_split(
    indices, test_size=0.2, random_state=1, stratify=labels
)
# Second split: halve the temp set into 10 % val + 10 % test
val_idx, test_idx = train_test_split(
    temp_idx, test_size=0.5, random_state=1, stratify=labels[temp_idx]
)

df_train = df.iloc[train_idx]
df_val = df.iloc[val_idx]
df_test = df.iloc[test_idx]

df_train

,isfake,content
101,1,puerto rico homeown brace anoth disast foreclo...
15820,0,<NUM> gun surrend birmingham gun buyback progr...
2570,0,new program introduc trump administr seem insp...
3745,0,former darpa director regina dugan discuss art...
10672,1,true blue conserv report contributor profil st...
...,...,...
3904,1,las cruce nm new mexico man su citi las cruce ...
12055,1,debt insan anyon washington even care <NUM> tr...
2041,1,maxim person brand job hunt reader think stori...
545,0,gold coast via wyong madeenati craig brennan -...


In [5]:
# Given a pandas or polars dataframe and potentially an number of tokens 
# to be included as features, returns a bag of words representation of
# the input's column 'content' as a scipy.sparse.csr_matrix.
def bow(df: pd.DataFrame | pl.DataFrame,
        vectorizer,
        save_as: None | str = None,
        fitting=True):

    # Create BoW using the CountVectorizer module
    # Define corpus using polars for efficiency
    if type(df) == pd.DataFrame:
        pl_series = pl.from_pandas(df['content'])
    else:
        pl_series = df.select('content')

    # Concat natural language cols to single list
    docs = pl_series.to_list()

    # Fit vectorizer to create compressed sparse column matrix for RAM efficiency
    if fitting:
        X = vectorizer.fit_transform(docs)
    else:
        X = vectorizer.transform(docs)

    if save_as: scipy.sparse.save_npz(save_as, X)
    
    return X

In [17]:
# Create BoW representation of trainig, validation and test set

# vectorizer
vec = CountVectorizer(max_features=int(1e4), ngram_range=(1,3))

X_train = bow(df_train, vec, save_as='./data/X_train.npz')
X_val = bow(df_val, vec, save_as='./data/X_val.npz', fitting=False)
X_test = bow(df_test, vec, save_as='./data/X_test.npz', fitting=False)

y_train = df_train['isfake']
y_val = df_val['isfake']
y_test = df_test['isfake']

In [18]:
classifier = LogisticRegression(random_state=1000, max_iter=2500)
classifier.fit(X_train, y_train)

preds = classifier.predict(X_val)
score = f1_score(y_val, preds)

print(classification_report(preds, y_val))

              precision    recall  f1-score   support

           0       0.87      0.89      0.88       771
           1       0.87      0.85      0.86       715

    accuracy                           0.87      1486
   macro avg       0.87      0.87      0.87      1486
weighted avg       0.87      0.87      0.87      1486



### Task 1 - grouping the labels
To group the labels in two groups, we first identified the unique labels used. This also guarantees that the drop before went as wished.

In [11]:
print(' '.join(sorted(list(set(y)))))

conspiracy fake hate junksci political reliable rumor


In [17]:
# Group labels in two groups. This is partly subjective which might be a problem later.
# We do this by specifying reliable labels and let all other labels be defined as fake.
# real_labs = ['political', 'satire', 'reliable', 'bias']
real_labs = ['reliable', 'political']

raw_ys = [y_train_raw, y_val_raw, y_test_raw]
y_train, y_val, y_test = [], [], []
ys = [y_train, y_val, y_test]

# We now transform the ys to be binary: 1 for fake, 0 for real.
for i, raw_y in enumerate(raw_ys):
    for j, lab in enumerate(raw_y):
        if lab in real_labs:
            ys[i].append(0)
        else:
            ys[i].append(1)

pd.concat([pd.Series(y_train_raw, name='label'), pd.Series(y_train, name='fake')], axis=1).head(20)

,label,fake
0,fake,1
1,political,0
2,conspiracy,1
3,conspiracy,1
4,reliable,0
5,hate,1
6,fake,1
7,political,0
8,reliable,0
9,political,0


### Task 2
To implement the logistic regression, we needed to extract the features from the training and test data. However, we ran into a problem.

If we used a traditional pandas or polars DataFrame, we would use too much RAM. We have 995,000 rows and using the top 10,000 words of the vocabulary with standard Int64 representation (8 bytes), this would take

995,000 * 10,000 * 8 bytes 
= 9.95 * 8 * 10^9 bytes 
= 79.6 * 10^9 bytes
= 79.6 GB

of RAM. This is absurd. However, we could expect that most of the BoW data are just zeros. Therefore, we might represent it using a sparse matrix as fx. a Compressed Sparse Row (CSR) matrix. This can be done efficiently using the CountVectorizer from sklearn.

In [18]:
# Given a pandas or polars dataframe and potentially an number of tokens 
# to be included as features, returns a bag of words representation of
# the input's column 'content' as a scipy.sparse.csr_matrix.
def bow(df: pd.DataFrame | pl.DataFrame, 
        num_feats = 10**4,
        save_as: None | str = None):

    # Create BoW using the CountVectorizer module
    # Define corpus using polars for efficiency
    if type(df) == pd.DataFrame:
        pl_series = pl.from_pandas(df['content'])
    else:
        pl_series = df.select('content')

    # Concat natural language cols to single list
    docs = pl_series.to_list()

    # Initialize vectorizer
    vec = CountVectorizer(max_features=num_feats)

    # Fit vectorizer to create compressed sparse column matrix for RAM efficiency
    X = vec.fit_transform(docs)

    if save_as: scipy.sparse.save_npz(save_as, X)
    
    return X

In [19]:
# Create BoW representation of trainig, validation and test set
X_train = bow(X_train_raw, save_as='./data/X_train.npz')
X_val = bow(X_val_raw, save_as='./data/X_val.npz')
X_test = bow(X_test_raw, save_as='./data/X_test.npz')

We now implement the logistic regression and evaluate it with F1 score.

In [ ]:
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_val = scaler.transform(X_val)

classifier = LogisticRegression(random_state=1000, max_iter=2500)
classifier.fit(X_train, y_train)

preds = classifier.predict(X_val)
score = f1_score(y_val, preds)

print(f'F1 Score: {score:.3f}')

F1 Score: 0.418
